<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will compute a <b>stable hash</b> for rules and deduplicate a tiny rule library.
</div>

# S05 · Canonicalize (basic) & deduplicate rules


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: deterministic serialization → hash
- Hands-on: build tiny library and deduplicate


# Theory

To deduplicate rules, we need a deterministic representation.
Here we use a simple, teaching-friendly approach:
- sort nodes/edges and serialize attributes into a string
- hash the string (SHA-256 → short digest)

For large-scale work you'd use stronger graph canonicalization, but this is enough for Paper 1.


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
import hashlib, json

# Load all toy reactions and extract minimal rules (reuse S04 logic)
df = pd.read_csv("data/reactions_mapped.csv")

def mol_to_mapped_graph(m: Chem.Mol) -> nx.Graph:
    G = nx.Graph()
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            G.add_node(amap, symbol=a.GetSymbol(), charge=int(a.GetFormalCharge()), aromatic=bool(a.GetIsAromatic()))
    for b in m.GetBonds():
        ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
        if ai and aj and ai in G.nodes and aj in G.nodes:
            G.add_edge(min(ai,aj), max(ai,aj), order=int(b.GetBondTypeAsDouble()), aromatic=bool(b.GetIsAromatic()))
    return G

def changed_pairs_from_smiles(am_rxn_smiles: str):
    react, prod = am_rxn_smiles.split(">>")
    mR=Chem.MolFromSmiles(react); mP=Chem.MolFromSmiles(prod)
    def pairs(m):
        out=set()
        for b in m.GetBonds():
            ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
            if ai and aj:
                out.add((min(ai,aj), max(ai,aj)))
        return out
    pR=pairs(mR); pP=pairs(mP)
    created = pP-pR
    deleted = pR-pP
    return mR, mP, created, deleted

def extract_min_rule(am_rxn_smiles: str) -> dict:
    mR, mP, created, deleted = changed_pairs_from_smiles(am_rxn_smiles)
    center_atoms = set([x for ab in (created | deleted) for x in ab])
    GR = mol_to_mapped_graph(mR); GP = mol_to_mapped_graph(mP)
    L = GR.subgraph(center_atoms).copy()
    R = GP.subgraph(center_atoms).copy()
    K_nodes = set(L.nodes()).intersection(R.nodes())
    K = L.subgraph(K_nodes).copy()
    return {
        "created": sorted(list(created)),
        "deleted": sorted(list(deleted)),
        "L": nx.node_link_data(L),
        "K": nx.node_link_data(K),
        "R": nx.node_link_data(R),
    }

rules = []
for _, r in df.iterrows():
    rules.append({"rxn_id": r.rxn_id, "label": r.label, **extract_min_rule(r.am_rxn_smiles)})
len(rules), rules[0].keys()


In [ ]:
def _graph_sig(nld: dict) -> str:
    nodes = sorted([(n["id"], n.get("symbol"), n.get("charge",0), n.get("aromatic",False)) for n in nld["nodes"]],
                   key=lambda x: (str(x[1]), int(x[2]), bool(x[3]), int(x[0])))
    edges = sorted([(min(e["source"], e["target"]), max(e["source"], e["target"]), e.get("order",0), e.get("aromatic",False))
                    for e in nld.get("links", [])],
                   key=lambda x: (x[0], x[1], x[2], x[3]))
    return "N|" + "|".join([f"{i}:{sym}:{chg}:{aro}" for i,sym,chg,aro in nodes]) +            ";E|" + "|".join([f"{u}-{v}:{o}:{aro}" for u,v,o,aro in edges])

def rule_hash(rule: dict) -> str:
    s = "L:" + _graph_sig(rule["L"]) + "||K:" + _graph_sig(rule["K"]) + "||R:" + _graph_sig(rule["R"])
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

for r in rules:
    r["rule_id"] = rule_hash(r)

unique = {}
for r in rules:
    unique.setdefault(r["rule_id"], r)

print("Total rules:", len(rules))
print("Unique rules:", len(unique))
list(unique.keys())


In [ ]:
# Export deduplicated rules to JSONL
rules_path = OUT / "S05_rules_dedup.jsonl"
with rules_path.open("w", encoding="utf-8") as f:
    for rid, r in unique.items():
        f.write(json.dumps(r) + "\n")
print("Wrote", rules_path)


# Discussion
- Naive signatures can collide: two different graphs might serialize similarly.
- Still, hashing is extremely useful to remove exact duplicates and to track provenance.


# Quiz
1. Why might two different graphs share the same naive signature?
2. Which node/edge attributes would you include to reduce collisions?
3. Export a deduplicated `rules.jsonl` with one JSON per rule.


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
